# Sharing Jobs with dapi

Share an existing job (read-only) with collaborators or a whole DesignSafe project team, view a job that was shared with you, and revoke grants.

**Requires dapi >= 0.5.5** (currently on the `dev` branch).

In [ ]:
%pip install --user --upgrade --quiet "git+https://github.com/DesignSafe-CI/dapi.git@dev"

In [ ]:
import dapi
from dapi import DSClient

print(
    f"dapi version: {dapi.__version__}"
)  # needs >= 0.5.5; restart kernel after upgrading

ds = DSClient()

## Owner side: share a job

Any job you own works — no need to run a new one.

In [ ]:
df = ds.jobs.list(limit=5)
df[["name", "uuid", "status", "appId", "created_dt"]]

In [ ]:
# Use the most recent job, or paste a specific UUID
job = ds.jobs.job(df.iloc[0]["uuid"])
print(f"{job.uuid}: {job.status}")

### Share with a user

The username is validated against the tenant before any grant is issued — a typo raises an error and shares nothing. The default grant covers job history, inputs, outputs, and the resubmit request (READ only).

In [ ]:
job.share(user_id="parduino")

In [ ]:
# Current grants on this job (empty DataFrame = not shared)
job.shares

### Share with a whole project team

Resolves every member (PI, co-PIs, team) of a DesignSafe project — preview with `ds.projects.members` first.

In [ ]:
# ds.projects.members("PRJ-XXXX")
# job.share(project_id="PRJ-XXXX")

## Grantee side: view a shared job

Run the cells below as the *collaborator* the job was shared with.

### Discover jobs shared with you

In [ ]:
shared = ds.jobs.list(list_type="SHARED_JOBS")
shared[["name", "uuid", "status", "appId", "created_dt"]] if len(shared) else shared

### Open and inspect a shared job

A shared job opens with the same `SubmittedJob` interface as your own jobs — status, details, history, runtime summary, even live monitoring while it runs. If the owner sent you a UUID directly, skip discovery and pass it to `ds.jobs.job(uuid)`.

In [ ]:
if len(shared):
    shared_job = ds.jobs.job(shared.iloc[0]["uuid"])
    print(f"{shared_job.uuid}: {shared_job.status}")
    print(f"Owner: {shared_job.details.owner}  App: {shared_job.details.appId}")
    shared_job.print_runtime_summary(verbose=False)
else:
    print("No jobs shared with this account yet.")

### Access outputs of a shared job

Share grants work through the Tapis *jobs* service. dapi's output methods (`get_results`, `get_output_content`, ...) currently read through the *files* service, which does not see job shares — so a grantee has three routes to outputs for now:

1. the share-aware Tapis job-output endpoints below (work regardless of where the job archived),
2. the DesignSafe portal: Workspace > Tools & Applications > Job Status,
3. the normal dapi methods, when the job archived to a shared project you belong to.

Routing dapi's output methods through the share-aware endpoints is planned.

In [ ]:
if len(shared):
    # List the shared job's outputs via the jobs service
    for item in ds.tapis.jobs.getJobOutputList(jobUuid=shared_job.uuid, outputPath="/"):
        print(f"- {item.name} ({item.type})")

    # Download a specific output, e.g. the SimCenter results bundle:
    # content = ds.tapis.jobs.getJobOutputDownload(
    #     jobUuid=shared_job.uuid, outputPath="results.zip"
    # )
    # with open("results.zip", "wb") as f:
    #     f.write(content)

## Owner side: revoke

In [ ]:
job.unshare(user_id="parduino")
job.shares